# Monitoramento da Qualidade da Água (NDCI - Reservatório do Funil)
Visualização interativa via Google Earth Engine utilizando a coleção harmonizada Sentinel-2 (`COPERNICUS/S2_SR_HARMONIZED`), com máscara óptica de água (`SCL`, `NDWI > 0.05`) e bloqueio duplo de nuvens (`QA60` + `MSK_CLDPRB < 20%`).

In [ ]:
import os
import ee
import geemap

# Inicializa o Google Earth Engine (configure GEE_PROJECT_ID no .env)
gee_project = os.getenv("GEE_PROJECT_ID", "seu-projeto-gee")
try:
    ee.Initialize(project=gee_project)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=gee_project)

# Coordenadas oficiais do espelho d'água do Reservatório do Funil (Resende/Itatiaia - RJ)
lat = -22.530116459790744
lon = -44.5639148196765
ponto_central = ee.Geometry.Point([lon, lat])
regiao_interesse = ponto_central.buffer(175)

Map = geemap.Map(center=[lat, lon], zoom=14)

def calcular_indices_agua_limpa(image):
    qa = image.select("QA60")
    qa_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    cldprb_mask = image.select("MSK_CLDPRB").lt(20)
    scl = image.select("SCL")
    scl_mask = scl.eq(6).Or(scl.eq(4)).Or(scl.eq(5))
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    water_mask = ndwi.gt(0.05)
    denom_mask = image.select("B5").add(image.select("B4")).gt(100)
    valid_mask = qa_mask.And(cldprb_mask).And(scl_mask).And(water_mask).And(denom_mask)
    ndci = image.normalizedDifference(["B5", "B4"]).clamp(-1.0, 1.0).rename("NDCI")
    return image.addBands([ndci, ndwi]).updateMask(valid_mask)

colecao_s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(regiao_interesse)
    .filterDate("2026-01-01", "2026-10-18")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
)

colecao_com_indices = colecao_s2.map(calcular_indices_agua_limpa)
imagem_media = colecao_com_indices.mean()

vis_rgb = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}
vis_ndci = {
    "min": -0.1,
    "max": 0.3,
    "bands": ["NDCI"],
    "palette": ["blue", "cyan", "green", "yellow", "red"]
}

Map.addLayer(imagem_media.clip(regiao_interesse), vis_rgb, "Sentinel-2 RGB (Água Limpa)")
Map.addLayer(imagem_media.select("NDCI").clip(regiao_interesse), vis_ndci, "Índice de Clorofila (NDCI)")
Map
